# Linked List

**Reference**
- The `ListNode` class (+ list <-> linked list helpers)

**Problems**
- LC 206 - Reverse Linked List
- LC 141 - Linked List Cycle
- LC 21 - Merge Two Sorted Lists
- LC 146 - LRU Cache

## Reference

A node holds a value and a pointer to the next node; the list *is* its head node, and
`None` marks the end. No indexing - reaching position `i` costs O(i) - but inserting or deleting given a node is O(1), which is the whole point.

Two habits that cover most problems:
- **Dummy head** - `dummy = node = ListNode()`, build off it, `return dummy.next`. Kills
  the "is this the first node?" special case.
- **Fast / slow pointers** - middle, nth-from-end, cycle detection.

Always save `curr.next` before you overwrite it.

In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def link(vals):
    """[1,2,3] -> 1->2->3"""
    dummy = curr = ListNode()
    for v in vals:
        curr.next = ListNode(v)
        curr = curr.next
    return dummy.next


def unlink(head):
    """1->2->3 -> [1,2,3]"""
    res = []
    while head:
        res.append(head.val)
        head = head.next
    return res

## Reverse Linked List (LC 206)

In [ ]:
def reverseList(head):
    prev, curr = None, head

    while curr:
        nxt = curr.next
        curr.next = prev

        prev = curr
        curr = nxt

    return prev

## Linked List Cycle (LC 141)

In [ ]:
def hasCycle(head):
    slow = fast = head

    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next

        if slow is fast:   # laps the slow pointer => cycle
            return True

    return False

## Merge Two Sorted Lists (LC 21)

In [ ]:
def mergeTwoLists(list1, list2):
    dummy = node = ListNode()

    while list1 and list2:
        if list1.val < list2.val:
            node.next = list1
            list1 = list1.next
        else:
            node.next = list2
            list2 = list2.next
        node = node.next

    node.next = list1 or list2   # tail of whichever is left

    return dummy.next

## LRU Cache (LC 146)

In [ ]:
class Node:
    """Doubly linked - needs prev to unlink in O(1)."""
    def __init__(self, key, val):
        self.key, self.val = key, val
        self.prev = self.next = None


class LRUCache:
    def __init__(self, capacity):
        self.cap = capacity
        self.cache = {}                              # key -> Node

        # dummy ends: left.next = LRU, right.prev = most recent
        self.left, self.right = Node(0, 0), Node(0, 0)
        self.left.next, self.right.prev = self.right, self.left

    def remove(self, node):
        prev, nxt = node.prev, node.next
        prev.next, nxt.prev = nxt, prev

    def insert(self, node):
        """Insert at the right end = most recently used."""
        prev, nxt = self.right.prev, self.right
        prev.next = nxt.prev = node
        node.prev, node.next = prev, nxt

    def get(self, key):
        if key not in self.cache:
            return -1

        self.remove(self.cache[key])                 # move to most recent
        self.insert(self.cache[key])
        return self.cache[key].val

    def put(self, key, value):
        if key in self.cache:
            self.remove(self.cache[key])

        self.cache[key] = Node(key, value)
        self.insert(self.cache[key])

        if len(self.cache) > self.cap:
            lru = self.left.next                     # evict from the left
            self.remove(lru)
            del self.cache[lru.key]